# Study report: thread-to-thread

The study measures the thread to thread wake latency (M3): the time from
a wake call in one thread to the first instruction of the woken thread.
It uses the app `thread_latency`, each kernel object of the app, and the
legacy and the new (`USE_SWITCH=y`) context switch. The study
`zephyr-versions` owns both experiments. This study reads them with
imports.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-default-latencies@nucleo_g474re/st-thread-to-thread"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## The modes of a histogram

A mode is a group of bins that each hold at least 0.1 % of the samples,
with at most 2 cycles between neighbours. Mode 1 is the fastest.
`mode_share(k)` of the checks is the share of the samples in mode k
(`hypotheses.yaml`, `scripts/benchmarks/studylib/stats.py`). The mode
tables use the same rule (`nbfig.modes`), so they show what the checks
see. This rule reads the fine bins only. Where a table has the columns
"... with coarse bins", these columns use the same rule over the fine
and the coarse bins: a coarse bin of 32 cycles is one bin. A difference
between the two views is a mode that the checks do not see.

The app data keep only the bins with samples. In the histograms, the
line goes down to a floor at each end of a group of bins, and an empty
bin is a gap.

## Kernel objects (thread-baseline-1m)

The ISR wakes the giver thread (priority 2) at a random time. The giver
wakes the taker through the kernel object. `t2t` is the time from the
wake call to the first instruction of the taker. The taker has priority 1
and preempts the giver. In the points `sem-nopreempt` and
`mutex-nopreempt` (`LATBENCH_T2T_PREEMPT: n`), the taker has priority 3:
it runs when the giver blocks again, so `t2t` includes the rest of the
giver loop (app page of `thread_latency`). The central result comes from
the point `sem`.

The stimulus comes every 50..150 us, not every 100..500 us as in the M2
experiments. The menu selects the metric: `t2t_ret` is the wake call to
its return, `isr2thr` is the ISR to the giver.

In [ ]:
if nbfig.built(inst, "thread-baseline-1m"):
    base = inst.rows("thread-baseline-1m")
    nbfig.dots(base, "Thread to thread per kernel object (thread-baseline-1m)",
               stats=("p50", "p99", "p99.99", "max")).show()
    display(nbfig.pivot(base[base.metric == "t2t"], ["p50", "p99", "p99.99", "max", "mean"]))

## The two modes (thread-baseline-1m)

The table gives the modes of `t2t` for each point. The histograms have
2048 fine bins and 2048 coarse bins (`experiment.md`). A mode in the
coarse bins has the width of one coarse bin.

In [ ]:
if nbfig.built(inst, "thread-baseline-1m"):
    display(nbfig.mode_table(inst, ["thread-baseline-1m"], "t2t", coarse=True))

## H1: the two modes with USE_SWITCH=y (thread-switch)

With the legacy context switch, the thread switch uses PendSV. With
`USE_SWITCH=y`, a thread switch on Cortex-M is `arch_switch()` inline in
`z_swap()`, in thread mode, without PendSV. H1 says: with `USE_SWITCH=y`
the wake has the same two modes as with PendSV, about 20 % of the
samples without a SysTick reprogram and about 80 % with two. The basis:
the slice reset is in `update_cache()` and does not depend on the arch
interface.

The legacy points are the reference from the same session, with the
same settings as `sem` and `mutex` of `thread-baseline-1m`. The figure
gives the share of each mode. The dashed lines mark the 20 % and the
80 % of H1. The check H1 a is manual: the design gives no tolerance for
"about".

In [ ]:
import plotly.graph_objects as go

if nbfig.built(inst, "thread-switch"):
    # the slow mode is in the coarse bins: the figure gives the view with the coarse bins
    mt = nbfig.mode_table(inst, ["thread-switch"], "t2t", coarse=True)
    fig = go.Figure()
    for k in (1, 2, 3):
        if f"share {k} with coarse bins" in mt:
            fig.add_trace(go.Bar(x=mt.index.get_level_values("point"), y=mt[f"share {k} with coarse bins"].fillna(0),
                                 name=f"mode {k}", marker_color=nbfig.color(k - 1),
                                 customdata=mt[f"mode {k} with coarse bins"],
                                 hovertemplate="%{x}: %{y:.4f} (mode at %{customdata} cycles)<extra>mode " + str(k)
                                               + "</extra>"))
    for y in (0.2, 0.8):
        fig.add_hline(y=y, line_dash="dash", line_color="#6e6d66", annotation_text=f"H1: about {y:.0%}",
                      annotation_position="top left")
    nbfig.style(fig, "Share of each mode of t2t: legacy switch against USE_SWITCH", "point", "share of the samples",
                barmode="group", yaxis_range=[0, 1]).show()
    display(mt)

In [ ]:
if nbfig.built(inst, "thread-switch"):
    series = nbfig.hists_of(inst, "thread-switch")
    nbfig.hist(series, "Thread to thread near the modes (thread-switch)", ["t2t"],
               xrange=nbfig.mode_range(series, "t2t", 100, coarse=True)).show()

## H2: the cost of USE_SWITCH=y (thread-switch)

H2 says: the thread switch costs more with `USE_SWITCH=y` than with
PendSV. Thus the mean `t2t` with `USE_SWITCH=y` is larger, for the
semaphore (H2 a) and for the mutex (H2 b). The basis: in
`wake-anchors-1m`, `USE_SWITCH=y` adds about 280 cycles per ISR wake.
The column `switch - legacy` gives the difference of each statistic.

In [ ]:
if nbfig.built(inst, "thread-switch"):
    sw = inst.rows("thread-switch")
    sw = sw[sw.metric == "t2t"]
    fig = go.Figure()
    for k, arch in enumerate(dict.fromkeys(sw["axis.arch"])):
        sub = sw[sw["axis.arch"] == arch]
        fig.add_trace(go.Bar(x=sub["axis.primitive"], y=sub["mean"], name=arch, marker_color=nbfig.color(k),
                             hovertemplate="%{x}: mean %{y:.0f} cycles<extra>" + arch + "</extra>"))
    nbfig.style(fig, "Mean thread to thread latency: legacy against USE_SWITCH", "primitive",
                "mean t2t (cycles)", barmode="group").show()
    t = sw.pivot_table(index="axis.primitive", columns="axis.arch", values=["p50", "p99", "mean", "max"],
                       aggfunc="first", sort=False)
    for st in ("p50", "p99", "mean", "max"):
        t[(st, "switch - legacy")] = t[(st, "switch")] - t[(st, "legacy")]
    display(t[[(st, a) for st in ("p50", "p99", "mean", "max") for a in ("legacy", "switch", "switch - legacy")]])